# Simulación de Señales EEG

## ¿Qué es una señal EEG?

**EEG** (Electroencefalografía) es una técnica que mide la actividad eléctrica del cerebro mediante electrodos colocados en el cuero cabelludo. Las señales EEG son fundamentales para:

- Diagnóstico de epilepsia y otros trastornos neurológicos
- Estudios del sueño
- Interfaces cerebro-computadora (BCI)
- Investigación en neurociencia cognitiva

## Bandas de Frecuencia del EEG

Las señales EEG se caracterizan por diferentes **bandas de frecuencia**, cada una asociada con diferentes estados cerebrales:

| Banda | Frecuencia (Hz) | Estado Mental Asociado |
|-------|-----------------|------------------------|
| **Delta (δ)** | 0.5 - 4 | Sueño profundo |
| **Theta (θ)** | 4 - 8 | Meditación, somnolencia |
| **Alpha (α)** | 8 - 13 | Relajación, ojos cerrados |
| **Beta (β)** | 13 - 30 | Concentración, actividad mental |
| **Gamma (γ)** | 30 - 100 | Procesamiento cognitivo complejo |

## Relación con Álgebra Lineal

Las señales EEG pueden entenderse como **combinaciones lineales** de ondas sinusoidales de diferentes frecuencias:

$$
s(t) = \sum_{i=1}^{n} A_i \sin(2\pi f_i t + \phi_i)
$$

Donde:
- $A_i$ = amplitud de la componente $i$
- $f_i$ = frecuencia de la componente $i$
- $\phi_i$ = fase de la componente $i$
- $t$ = tiempo

Cada señal sinusoidal puede verse como un **vector base** en el espacio de señales, y la señal EEG final es una combinación lineal de estos vectores base.

## Importar Librerías y Función Auxiliar

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq

# Importar la función de simulación
import sys
sys.path.append('../funciones_auxiliares')

# Copiar la función aquí (en un notebook real, se importaría)
def simularSenalEEG(duracion=10, fs=256, num_canales=1, incluir_bandas=['alpha', 'beta', 'theta', 'delta', 'gamma'], ruido=0.1, semilla=None):
    """
    Simular una señal EEG realista con múltiples bandas de frecuencia.
    
    Parámetros
    ----------
    duracion : float
        Duración de la señal en segundos (por defecto: 10)
    fs : int
        Frecuencia de muestreo en Hz (por defecto: 256)
    num_canales : int
        Número de canales EEG a simular (por defecto: 1)
    incluir_bandas : list
        Lista de bandas de frecuencia a incluir
    ruido : float
        Nivel de ruido a agregar a la señal (por defecto: 0.1)
    semilla : int o None
        Semilla para reproducibilidad
    """
    
    if semilla is not None:
        np.random.seed(semilla)
    
    bandas_freq = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 50)
    }
    
    bandas_amp = {
        'delta': 75,
        'theta': 40,
        'alpha': 50,
        'beta': 20,
        'gamma': 10
    }
    
    num_muestras = int(duracion * fs)
    t = np.linspace(0, duracion, num_muestras)
    
    if num_canales == 1:
        senal = np.zeros(num_muestras)
    else:
        senal = np.zeros((num_canales, num_muestras))
    
    info_bandas = {}
    
    for canal in range(num_canales):
        senal_canal = np.zeros(num_muestras)
        
        for banda in incluir_bandas:
            if banda in bandas_freq:
                freq_min, freq_max = bandas_freq[banda]
                amplitud_base = bandas_amp[banda]
                
                num_componentes = np.random.randint(2, 4)
                
                for _ in range(num_componentes):
                    freq = np.random.uniform(freq_min, freq_max)
                    amplitud = amplitud_base * np.random.uniform(0.5, 1.5)
                    fase = np.random.uniform(0, 2 * np.pi)
                    senal_canal += amplitud * np.sin(2 * np.pi * freq * t + fase)
                
                if banda not in info_bandas:
                    info_bandas[banda] = {
                        'rango_frecuencia': (freq_min, freq_max),
                        'amplitud_tipica': amplitud_base
                    }
        
        if ruido > 0:
            ruido_senal = np.random.normal(0, ruido * np.std(senal_canal), num_muestras)
            senal_canal += ruido_senal
        
        if num_canales == 1:
            senal = senal_canal
        else:
            senal[canal, :] = senal_canal
    
    return t, senal, info_bandas

print("Librerías importadas correctamente")

## Ejemplo 1: Simulación Básica de una Señal EEG

Vamos a generar una señal EEG simple de 10 segundos con todas las bandas de frecuencia.

In [ ]:
# Generar señal EEG
duracion = 10  # segundos
fs = 256  # frecuencia de muestreo en Hz

t, senal, info = simularSenalEEG(
    duracion=duracion,
    fs=fs,
    num_canales=1,
    incluir_bandas=['alpha', 'beta', 'theta', 'delta', 'gamma'],
    ruido=0.1,
    semilla=42
)

# Información de la señal
print(f"Duración: {duracion} segundos")
print(f"Frecuencia de muestreo: {fs} Hz")
print(f"Número de muestras: {len(senal)}")
print(f"\nBandas incluidas:")
for banda, datos in info.items():
    print(f"  - {banda.capitalize()}: {datos['rango_frecuencia'][0]}-{datos['rango_frecuencia'][1]} Hz")

In [ ]:
# Visualizar la señal completa
plt.figure(figsize=(14, 5))
plt.plot(t, senal, linewidth=0.8)
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel('Amplitud (μV)', fontsize=12)
plt.title('Señal EEG Simulada - 10 segundos', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Visualizar un segmento más pequeño (primeros 2 segundos)
plt.figure(figsize=(14, 5))
idx = t <= 2  # primeros 2 segundos
plt.plot(t[idx], senal[idx], linewidth=1)
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel('Amplitud (μV)', fontsize=12)
plt.title('Señal EEG Simulada - Detalle (2 segundos)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Ejemplo 2: Análisis de Frecuencias (Transformada de Fourier)

Utilizamos la **Transformada de Fourier** para analizar el contenido de frecuencias de la señal EEG. Esto nos permite ver qué frecuencias están presentes en la señal.

In [ ]:
# Calcular la Transformada de Fourier
N = len(senal)
yf = fft(senal)
xf = fftfreq(N, 1/fs)[:N//2]

# Calcular el espectro de potencia
potencia = 2.0/N * np.abs(yf[0:N//2])

# Graficar el espectro de frecuencias
plt.figure(figsize=(14, 5))
plt.plot(xf, potencia, linewidth=1)
plt.xlabel('Frecuencia (Hz)', fontsize=12)
plt.ylabel('Potencia', fontsize=12)
plt.title('Espectro de Frecuencias de la Señal EEG', fontsize=14, fontweight='bold')
plt.xlim(0, 60)  # Enfocarse en las frecuencias relevantes
plt.grid(True, alpha=0.3)

# Marcar las bandas de frecuencia
bandas_rangos = {
    'Delta': (0.5, 4, 'blue'),
    'Theta': (4, 8, 'green'),
    'Alpha': (8, 13, 'orange'),
    'Beta': (13, 30, 'red'),
    'Gamma': (30, 50, 'purple')
}

for banda, (f_min, f_max, color) in bandas_rangos.items():
    plt.axvspan(f_min, f_max, alpha=0.1, color=color, label=f'{banda} ({f_min}-{f_max} Hz)')

plt.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

## Ejemplo 3: Señales EEG con Diferentes Estados Mentales

Simulamos señales EEG que representan diferentes estados mentales variando las bandas de frecuencia presentes.

In [ ]:
# Simular diferentes estados mentales
estados = {
    'Relajación (Alpha dominante)': ['alpha', 'theta'],
    'Concentración (Beta dominante)': ['beta', 'alpha'],
    'Sueño profundo (Delta dominante)': ['delta', 'theta'],
    'Procesamiento cognitivo (Gamma/Beta)': ['gamma', 'beta']
}

fig, axes = plt.subplots(4, 1, figsize=(14, 12))

for idx, (estado, bandas) in enumerate(estados.items()):
    # Generar señal
    t_estado, senal_estado, _ = simularSenalEEG(
        duracion=3,
        fs=256,
        num_canales=1,
        incluir_bandas=bandas,
        ruido=0.1,
        semilla=42 + idx
    )
    
    # Graficar
    axes[idx].plot(t_estado, senal_estado, linewidth=0.8)
    axes[idx].set_ylabel('Amplitud (μV)', fontsize=10)
    axes[idx].set_title(estado, fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    
axes[-1].set_xlabel('Tiempo (s)', fontsize=12)
plt.tight_layout()
plt.show()

## Ejemplo 4: Múltiples Canales EEG

En aplicaciones reales, el EEG utiliza múltiples electrodos (canales) colocados en diferentes posiciones del cuero cabelludo. Vamos a simular un sistema con 4 canales.

In [ ]:
# Simular 4 canales EEG
num_canales = 4
t_multi, senal_multi, _ = simularSenalEEG(
    duracion=5,
    fs=256,
    num_canales=num_canales,
    incluir_bandas=['alpha', 'beta', 'theta', 'delta'],
    ruido=0.15,
    semilla=42
)

print(f"Forma de la señal multicanal: {senal_multi.shape}")
print(f"(canales × muestras)")

In [ ]:
# Visualizar los 4 canales
nombres_canales = ['Frontal (Fp1)', 'Central (C3)', 'Parietal (P3)', 'Occipital (O1)']

fig, axes = plt.subplots(num_canales, 1, figsize=(14, 10))

for i in range(num_canales):
    axes[i].plot(t_multi, senal_multi[i, :], linewidth=0.8)
    axes[i].set_ylabel('Amplitud (μV)', fontsize=10)
    axes[i].set_title(f'Canal {i+1}: {nombres_canales[i]}', fontsize=11, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(0, 5)

axes[-1].set_xlabel('Tiempo (s)', fontsize=12)
plt.tight_layout()
plt.show()

## Ejemplo 5: Álgebra Lineal - Matriz de Señales

Podemos representar las señales multicanal como una **matriz** donde:
- Cada **fila** es un canal EEG
- Cada **columna** es un punto temporal

Esta representación matricial permite aplicar operaciones de álgebra lineal para:
- Análisis de componentes principales (PCA)
- Separación de fuentes (ICA)
- Filtrado espacial
- Reducción de ruido

In [ ]:
# Matriz de señales EEG
print("Matriz de señales EEG:")
print(f"Forma: {senal_multi.shape}")
print(f"Cada fila representa un canal, cada columna un instante de tiempo")
print(f"\nPrimeras 5 muestras de cada canal:")
print(senal_multi[:, :5])

In [ ]:
# Calcular la matriz de covarianza entre canales
# Esto muestra cómo se relacionan las señales de diferentes canales
matriz_cov = np.cov(senal_multi)

print("Matriz de Covarianza entre canales:")
print(f"Forma: {matriz_cov.shape}")
print(f"\nMatriz de covarianza:")
print(matriz_cov)

# Visualizar la matriz de covarianza
plt.figure(figsize=(8, 6))
plt.imshow(matriz_cov, cmap='RdBu_r', interpolation='nearest')
plt.colorbar(label='Covarianza')
plt.title('Matriz de Covarianza entre Canales EEG', fontsize=14, fontweight='bold')
plt.xlabel('Canal', fontsize=12)
plt.ylabel('Canal', fontsize=12)
plt.xticks(range(num_canales), [f'C{i+1}' for i in range(num_canales)])
plt.yticks(range(num_canales), [f'C{i+1}' for i in range(num_canales)])

# Agregar valores en cada celda
for i in range(num_canales):
    for j in range(num_canales):
        text = plt.text(j, i, f'{matriz_cov[i, j]:.0f}',
                       ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

## Ejemplo 6: Espectrograma - Análisis Tiempo-Frecuencia

Un **espectrograma** muestra cómo cambia el contenido de frecuencias a lo largo del tiempo. Es una herramienta muy útil para analizar señales EEG no estacionarias.

In [ ]:
# Generar una señal que cambia con el tiempo
t_dinamic, senal_dinamica, _ = simularSenalEEG(
    duracion=10,
    fs=256,
    num_canales=1,
    incluir_bandas=['alpha', 'beta', 'theta', 'delta'],
    ruido=0.1,
    semilla=123
)

# Calcular el espectrograma
f, t_spec, Sxx = signal.spectrogram(senal_dinamica, fs, nperseg=256)

# Graficar el espectrograma
plt.figure(figsize=(14, 6))
plt.pcolormesh(t_spec, f, 10 * np.log10(Sxx), shading='gouraud', cmap='viridis')
plt.ylabel('Frecuencia (Hz)', fontsize=12)
plt.xlabel('Tiempo (s)', fontsize=12)
plt.title('Espectrograma de la Señal EEG', fontsize=14, fontweight='bold')
plt.colorbar(label='Potencia (dB)')
plt.ylim(0, 50)  # Enfocarse en frecuencias relevantes

# Agregar líneas para las bandas
plt.axhline(y=4, color='white', linestyle='--', alpha=0.5, linewidth=1)
plt.axhline(y=8, color='white', linestyle='--', alpha=0.5, linewidth=1)
plt.axhline(y=13, color='white', linestyle='--', alpha=0.5, linewidth=1)
plt.axhline(y=30, color='white', linestyle='--', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.show()

## Conclusiones

En este notebook hemos:

1. ✅ Aprendido qué son las señales EEG y sus bandas de frecuencia
2. ✅ Simulado señales EEG realistas usando combinaciones lineales de ondas sinusoidales
3. ✅ Analizado el contenido de frecuencias usando la Transformada de Fourier
4. ✅ Simulado diferentes estados mentales variando las bandas de frecuencia
5. ✅ Trabajado con señales multicanal representadas como matrices
6. ✅ Aplicado conceptos de álgebra lineal (matrices, covarianza)
7. ✅ Visualizado el contenido tiempo-frecuencia con espectrogramas

### Relación con Álgebra Lineal

Las señales EEG son un excelente ejemplo de aplicación práctica del álgebra lineal:

- **Vectores**: Cada señal temporal es un vector en un espacio de alta dimensión
- **Matrices**: Las señales multicanal forman matrices que podemos analizar
- **Combinaciones lineales**: Las señales son sumas ponderadas de ondas base
- **Transformaciones lineales**: Filtrado, PCA, ICA son transformaciones lineales
- **Espacios vectoriales**: Las señales forman un espacio vectorial con producto interno

### Aplicaciones Prácticas

- 🧠 **Neurociencia**: Estudiar la actividad cerebral
- 🏥 **Medicina**: Diagnóstico de epilepsia y trastornos del sueño
- 🎮 **BCI**: Interfaces cerebro-computadora para control de dispositivos
- 🔬 **Investigación**: Estudios sobre cognición y procesos mentales